# Accident Data KDE Heatmap Pipeline and Data Transformation

This notebook generates kernel density estimation (KDE) heatmaps for German accident data on a 3000m hexagonal grid and transform data in various forms for visualization purposes.

## Step 0: Load Accident Data

Load all accident sheets from the Excel file (2016-2024) into Spark DataFrames for processing.

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession

# Create a local Spark session
spark = SparkSession.builder.appName("AccidentData").getOrCreate()

# Read all sheets from Excel file into pandas dataframes
excel_file = "./public/accident_data_2016-2024.xlsx"
all_sheets = pd.read_excel(excel_file, sheet_name=None)

print(f"Sheet names: {list(all_sheets.keys())}")

# Convert each pandas dataframe to a Spark dataframe
spark_dataframes = []
for sheet_name, pdf in all_sheets.items():
    sdf = spark.createDataFrame(pdf)
    spark_dataframes.append(sdf)
    print(f"Loaded sheet '{sheet_name}' into Spark DataFrame with {pdf.shape[0]} rows, {pdf.shape[1]} columns")

print(f"\nTotal sheets loaded: {len(spark_dataframes)}")

## Step 1 :Generate the 1500m Hex Grid

This cell creates the base hexagonal grid used for the map visualizations.

### What it does
- Loads `germany_states.geo.json` and extracts all coordinates from the GeoJSON to find Germany’s bounding box
- Converts longitude/latitude into Web Mercator meters
- Builds a staggered hex grid with:
  - `HEX_RADIUS_M = 1500`
  - `HEX_WIDTH_M = sqrt(3) * HEX_RADIUS_M`
  - `ROW_STEP_M = 1.5 * HEX_RADIUS_M`
  - `COL_STEP_M = HEX_WIDTH_M`
- Converts the hex center positions back to longitude/latitude
- Keeps only hexes that fall within the Germany bounding box
- Saves the result to `./public/heatmaps/hex_centers_1500m.json`

In [ ]:
import json
import math
from pathlib import Path
import pandas as pd

# Generate the original 1500 m hex grid for Germany.
EARTH_RADIUS_M = 6378137.0
HEX_RADIUS_M = 1500.0
HEX_WIDTH_M = math.sqrt(3.0) * HEX_RADIUS_M
ROW_STEP_M = 1.5 * HEX_RADIUS_M
COL_STEP_M = HEX_WIDTH_M


def lonlat_to_mercator(lon, lat):
    lat = max(min(float(lat), 89.9999), -89.9999)
    lon = float(lon)
    x = EARTH_RADIUS_M * math.radians(lon)
    y = EARTH_RADIUS_M * math.log(math.tan(math.pi / 4.0 + math.radians(lat) / 2.0))
    return x, y


def mercator_to_lonlat(x, y):
    lon = math.degrees(x / EARTH_RADIUS_M)
    lat = math.degrees(2.0 * math.atan(math.exp(y / EARTH_RADIUS_M)) - math.pi / 2.0)
    return lon, lat


def collect_coordinates(value, collected):
    if isinstance(value, (list, tuple)):
        if len(value) == 2 and all(isinstance(item, (int, float)) for item in value):
            collected.append((float(value[0]), float(value[1])))
        else:
            for item in value:
                collect_coordinates(item, collected)
    elif isinstance(value, dict):
        for item in value.values():
            collect_coordinates(item, collected)


geojson_path = Path("./public/germany_states.geo.json")
if not geojson_path.exists():
    raise FileNotFoundError(f"Missing Germany geojson file: {geojson_path}")

with geojson_path.open("r", encoding="utf-8") as f:
    germany_geojson = json.load(f)

coordinates = []
collect_coordinates(germany_geojson, coordinates)

if not coordinates:
    raise ValueError("Could not extract any coordinates from germany_states.geo.json")

lon_values = [lon for lon, _ in coordinates]
lat_values = [lat for _, lat in coordinates]

bbox = {
    "lon_min": min(lon_values),
    "lon_max": max(lon_values),
    "lat_min": min(lat_values),
    "lat_max": max(lat_values),
}

x_min, y_min = lonlat_to_mercator(bbox["lon_min"], bbox["lat_min"])
x_max, y_max = lonlat_to_mercator(bbox["lon_max"], bbox["lat_max"])

x_start = x_min - COL_STEP_M
x_end = x_max + COL_STEP_M
y_start = y_min - ROW_STEP_M
y_end = y_max + ROW_STEP_M

cells = []
row = 0
y = y_start
while y <= y_end:
    x_offset = 0.5 * COL_STEP_M if row % 2 else 0.0
    col = 0
    x = x_start + x_offset
    while x <= x_end:
        lon, lat = mercator_to_lonlat(x, y)
        if (bbox["lon_min"] - 0.5) <= lon <= (bbox["lon_max"] + 0.5) and (bbox["lat_min"] - 0.5) <= lat <= (bbox["lat_max"] + 0.5):
            cells.append({
                "hex_row": row,
                "hex_col": col,
                "center_x_m": round(x, 3),
                "center_y_m": round(y, 3),
                "longitude": round(lon, 8),
                "latitude": round(lat, 8),
            })
        col += 1
        x += COL_STEP_M
    row += 1
    y += ROW_STEP_M

hex_1500_payload = {
    "hex_radius_m": HEX_RADIUS_M,
    "hex_width_m": HEX_WIDTH_M,
    "row_step_m": ROW_STEP_M,
    "col_step_m": COL_STEP_M,
    "bbox": bbox,
    "cells": cells,
}

out_hex_path = Path("./public/heatmaps/hex_centers_1500m.json")
with out_hex_path.open("w", encoding="utf-8") as f:
    json.dump(hex_1500_payload, f, ensure_ascii=True)

hex_1500_df = pd.DataFrame(cells)
print(f"Wrote {out_hex_path} with {len(cells)} cells")
print(hex_1500_df.head().to_string(index=False))

## Step 2: Generate 3000m Hex Grid

Create a coarser hexagonal grid with 3000m radius to emphasize broader regional patterns in accident density. Based on the existing 1500m grid metadata.

In [ ]:
import json
import math
from pathlib import Path
import pandas as pd

# Generate a coarser 3000 m hex grid for broader regional patterns.
EARTH_RADIUS_M = 6378137.0
HEX_RADIUS_M = 3000.0
HEX_WIDTH_M = math.sqrt(3.0) * HEX_RADIUS_M
ROW_STEP_M = 1.5 * HEX_RADIUS_M
COL_STEP_M = HEX_WIDTH_M

def lonlat_to_mercator(lon, lat):
    lat = max(min(float(lat), 89.9999), -89.9999)
    lon = float(lon)
    x = EARTH_RADIUS_M * math.radians(lon)
    y = EARTH_RADIUS_M * math.log(math.tan(math.pi / 4.0 + math.radians(lat) / 2.0))
    return x, y

def mercator_to_lonlat(x, y):
    lon = math.degrees(x / EARTH_RADIUS_M)
    lat = math.degrees(2.0 * math.atan(math.exp(y / EARTH_RADIUS_M)) - math.pi / 2.0)
    return lon, lat

base_hex_path = Path("./public/heatmaps/hex_centers_1500m.json")
if not base_hex_path.exists():
    raise FileNotFoundError(f"Missing base hex-center file: {base_hex_path}")

with base_hex_path.open("r", encoding="utf-8") as f:
    base_hex_data = json.load(f)

bbox = base_hex_data["bbox"]
lon_min = float(bbox["lon_min"] )
lon_max = float(bbox["lon_max"] )
lat_min = float(bbox["lat_min"] )
lat_max = float(bbox["lat_max"] )

x_min, y_min = lonlat_to_mercator(lon_min, lat_min)
x_max, y_max = lonlat_to_mercator(lon_max, lat_max)

x_start = x_min - COL_STEP_M
x_end = x_max + COL_STEP_M
y_start = y_min - ROW_STEP_M
y_end = y_max + ROW_STEP_M

cells = []
row = 0
y = y_start
while y <= y_end:
    x_offset = 0.5 * COL_STEP_M if row % 2 else 0.0
    col = 0
    x = x_start + x_offset
    while x <= x_end:
        lon, lat = mercator_to_lonlat(x, y)
        if (lon_min - 0.5) <= lon <= (lon_max + 0.5) and (lat_min - 0.5) <= lat <= (lat_max + 0.5):
            cells.append({
                "hex_row": row,
                "hex_col": col,
                "center_x_m": round(x, 3),
                "center_y_m": round(y, 3),
                "longitude": round(lon, 8),
                "latitude": round(lat, 8),
            })
        col += 1
        x += COL_STEP_M
    row += 1
    y += ROW_STEP_M

hex_3000_payload = {
    "hex_radius_m": HEX_RADIUS_M,
    "hex_width_m": HEX_WIDTH_M,
    "row_step_m": ROW_STEP_M,
    "col_step_m": COL_STEP_M,
    "bbox": bbox,
    "cells": cells,
}

out_hex_path = Path("./public/heatmaps/hex_centers_3000m.json")
with out_hex_path.open("w", encoding="utf-8") as f:
    json.dump(hex_3000_payload, f, ensure_ascii=True)

hex_3000_df = pd.DataFrame(cells)
print(f"Wrote {out_hex_path} with {len(cells)} cells")
print(hex_3000_df.head().to_string(index=False))

## Step 3.1.: Generate 2024 KDE Heatmap

Compute kernel density estimation for 2024 accidents on the 3000m hex grid. Uses a 3000m Gaussian bandwidth, severity-weighted contributions, and intensity thresholding to produce sparse, meaningful heatmap data.

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

# Generate the 2024 heatmap on the new 3000 m hex grid.
EARTH_RADIUS_M = 6378137.0
BANDWIDTH_M = 3000.0
CUTOFF_SIGMA = 3.0
CUTOFF_M = BANDWIDTH_M * CUTOFF_SIGMA
INTENSITY_THRESHOLD = 0.02

hex_path = Path("./public/heatmaps/hex_centers_3000m.json")
if not hex_path.exists():
    raise FileNotFoundError(f"Missing hex-center file: {hex_path}")

with hex_path.open("r", encoding="utf-8") as f:
    hex_data = json.load(f)

hex_centers_pdf = pd.DataFrame(hex_data["cells"])
hex_centers_sdf = spark.createDataFrame(hex_centers_pdf)

sheet_names = list(all_sheets.keys())
last_sheet_name = sheet_names[-1]
accident_source = spark.createDataFrame(all_sheets[last_sheet_name])

severity_weight_expr = (
    F.when(F.col("severity") == 1, F.lit(5.0))
     .when(F.col("severity") == 2, F.lit(3.0))
     .when(F.col("severity") == 3, F.lit(1.0))
     .otherwise(F.lit(1.0))
)

accident_2024_m = (
    accident_source
    .select("XGCSWGS84", "YGCSWGS84", "severity")
    .withColumn("longitude", F.col("XGCSWGS84").cast("double"))
    .withColumn("latitude", F.col("YGCSWGS84").cast("double"))
    .filter(F.col("longitude").isNotNull() & F.col("latitude").isNotNull())
    .withColumn("x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
    .withColumn("severity_weight", severity_weight_expr)
    .filter(F.col("severity_weight").isNotNull())
)

hex_centers_m = (
    hex_centers_sdf
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("center_x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("center_y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
)

hex_centers_broadcast = F.broadcast(hex_centers_m.select("hex_row", "hex_col", "longitude", "latitude", "center_x_m", "center_y_m"))

kde_contrib = (
    accident_2024_m.alias("a")
    .crossJoin(hex_centers_broadcast.alias("h"))
    .withColumn(
        "distance_m",
        F.sqrt(
            F.pow(F.col("a.x_m") - F.col("h.center_x_m"), 2) +
            F.pow(F.col("a.y_m") - F.col("h.center_y_m"), 2)
        )
    )
    .filter(F.col("distance_m") <= F.lit(CUTOFF_M))
    .withColumn(
        "distance_weight",
        F.exp(-(F.col("distance_m") * F.col("distance_m")) / F.lit(2.0 * BANDWIDTH_M * BANDWIDTH_M))
    )
    .withColumn("contribution", F.col("distance_weight") * F.col("a.severity_weight"))
)

raw_intensity = (
    kde_contrib
    .groupBy("h.hex_row", "h.hex_col", "h.longitude", "h.latitude", "h.center_x_m", "h.center_y_m")
    .agg(
        F.sum("contribution").alias("raw_intensity"),
        F.count(F.lit(1)).alias("support_points")
    )
)

max_raw = raw_intensity.agg(F.max("raw_intensity").alias("max_raw")).collect()[0]["max_raw"]
if max_raw is None or max_raw == 0:
    max_raw = 1.0

heatmap_2024 = (
    raw_intensity
    .withColumn("heatmap_intensity", F.col("raw_intensity") / F.lit(float(max_raw)))
    .orderBy(F.desc("raw_intensity"))
)

heatmap_2024_sparse = heatmap_2024.filter(F.col("heatmap_intensity") >= F.lit(INTENSITY_THRESHOLD))

rows = heatmap_2024_sparse.select("longitude", "latitude", "raw_intensity", "heatmap_intensity", "support_points").toLocalIterator()
records = [r.asDict(recursive=True) for r in rows]
payload = {"year": 2024, "bandwidth_m": BANDWIDTH_M, "intensity_threshold": INTENSITY_THRESHOLD, "cells": records}

out_path = "./public/heatmaps/heatmap_2024.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=True)

print(f"Wrote {out_path} with {len(records)} cells")
print("2024 heatmap bandwidth:", BANDWIDTH_M)

## Step 3.2.: Generate 2016 KDE Heatmap

Compute kernel density estimation for 2016 accidents on the 3000m hex grid using identical parameters as 2024 for consistency and comparability.

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

# Generate the 2016 heatmap on the new 3000 m hex grid.
EARTH_RADIUS_M = 6378137.0
BANDWIDTH_M = 3000.0
CUTOFF_SIGMA = 3.0
CUTOFF_M = BANDWIDTH_M * CUTOFF_SIGMA
INTENSITY_THRESHOLD = 0.02

hex_path = Path("./public/heatmaps/hex_centers_3000m.json")
if not hex_path.exists():
    raise FileNotFoundError(f"Missing hex-center file: {hex_path}")

with hex_path.open("r", encoding="utf-8") as f:
    hex_data = json.load(f)

hex_centers_pdf = pd.DataFrame(hex_data["cells"])
hex_centers_sdf = spark.createDataFrame(hex_centers_pdf)

sheet_names = list(all_sheets.keys())
first_sheet_name = sheet_names[0]
accident_source = spark.createDataFrame(all_sheets[first_sheet_name])

severity_weight_expr = (
    F.when(F.col("severity") == 1, F.lit(5.0))
     .when(F.col("severity") == 2, F.lit(3.0))
     .when(F.col("severity") == 3, F.lit(1.0))
     .otherwise(F.lit(1.0))
)

accident_2016_m = (
    accident_source
    .select("XGCSWGS84", "YGCSWGS84", "severity")
    .withColumn("longitude", F.col("XGCSWGS84").cast("double"))
    .withColumn("latitude", F.col("YGCSWGS84").cast("double"))
    .filter(F.col("longitude").isNotNull() & F.col("latitude").isNotNull())
    .withColumn("x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
    .withColumn("severity_weight", severity_weight_expr)
    .filter(F.col("severity_weight").isNotNull())
)

hex_centers_m = (
    hex_centers_sdf
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("center_x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("center_y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
)

hex_centers_broadcast = F.broadcast(hex_centers_m.select("hex_row", "hex_col", "longitude", "latitude", "center_x_m", "center_y_m"))

kde_contrib = (
    accident_2016_m.alias("a")
    .crossJoin(hex_centers_broadcast.alias("h"))
    .withColumn(
        "distance_m",
        F.sqrt(
            F.pow(F.col("a.x_m") - F.col("h.center_x_m"), 2) +
            F.pow(F.col("a.y_m") - F.col("h.center_y_m"), 2)
        )
    )
    .filter(F.col("distance_m") <= F.lit(CUTOFF_M))
    .withColumn(
        "distance_weight",
        F.exp(-(F.col("distance_m") * F.col("distance_m")) / F.lit(2.0 * BANDWIDTH_M * BANDWIDTH_M))
    )
    .withColumn("contribution", F.col("distance_weight") * F.col("a.severity_weight"))
)

raw_intensity = (
    kde_contrib
    .groupBy("h.hex_row", "h.hex_col", "h.longitude", "h.latitude", "h.center_x_m", "h.center_y_m")
    .agg(
        F.sum("contribution").alias("raw_intensity"),
        F.count(F.lit(1)).alias("support_points")
    )
)

max_raw = raw_intensity.agg(F.max("raw_intensity").alias("max_raw")).collect()[0]["max_raw"]
if max_raw is None or max_raw == 0:
    max_raw = 1.0

heatmap_2016 = (
    raw_intensity
    .withColumn("heatmap_intensity", F.col("raw_intensity") / F.lit(float(max_raw)))
    .orderBy(F.desc("raw_intensity"))
)

heatmap_2016_sparse = heatmap_2016.filter(F.col("heatmap_intensity") >= F.lit(INTENSITY_THRESHOLD))

rows = heatmap_2016_sparse.select("longitude", "latitude", "raw_intensity", "heatmap_intensity", "support_points").toLocalIterator()
records = [r.asDict(recursive=True) for r in rows]
payload = {"year": 2016, "bandwidth_m": BANDWIDTH_M, "intensity_threshold": INTENSITY_THRESHOLD, "cells": records}

out_path = "./public/heatmaps/heatmap_2016.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=True)

print(f"Wrote {out_path} with {len(records)} cells")
print("2016 heatmap bandwidth:", BANDWIDTH_M)

## Step 3.3.: Generate 2017 KDE Heatmap

Compute kernel density estimation for 2017 accidents on the 3000m hex grid using identical parameters as 2024 and 2016 for consistency and comparability.

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

# Generate the 2017 heatmap on the new 3000 m hex grid.
EARTH_RADIUS_M = 6378137.0
BANDWIDTH_M = 3000.0
CUTOFF_SIGMA = 3.0
CUTOFF_M = BANDWIDTH_M * CUTOFF_SIGMA
INTENSITY_THRESHOLD = 0.02

hex_path = Path("./public/heatmaps/hex_centers_3000m.json")
if not hex_path.exists():
    raise FileNotFoundError(f"Missing hex-center file: {hex_path}")

with hex_path.open("r", encoding="utf-8") as f:
    hex_data = json.load(f)

hex_centers_pdf = pd.DataFrame(hex_data["cells"])
hex_centers_sdf = spark.createDataFrame(hex_centers_pdf)

sheet_names = list(all_sheets.keys())
accident_source = spark.createDataFrame(all_sheets[sheet_names[1]])

severity_weight_expr = (
    F.when(F.col("severity") == 1, F.lit(5.0))
     .when(F.col("severity") == 2, F.lit(3.0))
     .when(F.col("severity") == 3, F.lit(1.0))
     .otherwise(F.lit(1.0))
)

accident_2017_m = (
    accident_source
    .select("XGCSWGS84", "YGCSWGS84", "severity")
    .withColumn("longitude", F.col("XGCSWGS84").cast("double"))
    .withColumn("latitude", F.col("YGCSWGS84").cast("double"))
    .filter(F.col("longitude").isNotNull() & F.col("latitude").isNotNull())
    .withColumn("x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
    .withColumn("severity_weight", severity_weight_expr)
    .filter(F.col("severity_weight").isNotNull())
)

hex_centers_m = (
    hex_centers_sdf
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("center_x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("center_y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
)

hex_centers_broadcast = F.broadcast(hex_centers_m.select("hex_row", "hex_col", "longitude", "latitude", "center_x_m", "center_y_m"))

kde_contrib = (
    accident_2017_m.alias("a")
    .crossJoin(hex_centers_broadcast.alias("h"))
    .withColumn(
        "distance_m",
        F.sqrt(
            F.pow(F.col("a.x_m") - F.col("h.center_x_m"), 2) +
            F.pow(F.col("a.y_m") - F.col("h.center_y_m"), 2)
        )
    )
    .filter(F.col("distance_m") <= F.lit(CUTOFF_M))
    .withColumn(
        "distance_weight",
        F.exp(-(F.col("distance_m") * F.col("distance_m")) / F.lit(2.0 * BANDWIDTH_M * BANDWIDTH_M))
    )
    .withColumn("contribution", F.col("distance_weight") * F.col("a.severity_weight"))
)

raw_intensity = (
    kde_contrib
    .groupBy("h.hex_row", "h.hex_col", "h.longitude", "h.latitude", "h.center_x_m", "h.center_y_m")
    .agg(
        F.sum("contribution").alias("raw_intensity"),
        F.count(F.lit(1)).alias("support_points")
    )
)

max_raw = raw_intensity.agg(F.max("raw_intensity").alias("max_raw")).collect()[0]["max_raw"]
if max_raw is None or max_raw == 0:
    max_raw = 1.0

heatmap_2017 = (
    raw_intensity
    .withColumn("heatmap_intensity", F.col("raw_intensity") / F.lit(float(max_raw)))
    .orderBy(F.desc("raw_intensity"))
)

heatmap_2017_sparse = heatmap_2017.filter(F.col("heatmap_intensity") >= F.lit(INTENSITY_THRESHOLD))

rows = heatmap_2017_sparse.select("longitude", "latitude", "raw_intensity", "heatmap_intensity", "support_points").toLocalIterator()
records = [r.asDict(recursive=True) for r in rows]
payload = {"year": 2017, "bandwidth_m": BANDWIDTH_M, "intensity_threshold": INTENSITY_THRESHOLD, "cells": records}

out_path = "./public/heatmaps/heatmap_2017.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=True)

print(f"Wrote {out_path} with {len(records)} cells")
print("2017 heatmap bandwidth:", BANDWIDTH_M)

## Step 3.4.: Generate 2018 KDE Heatmap

Compute kernel density estimation for 2018 accidents on the 3000m hex grid using identical parameters for consistency across all years.

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

EARTH_RADIUS_M = 6378137.0
BANDWIDTH_M = 3000.0
CUTOFF_SIGMA = 3.0
CUTOFF_M = BANDWIDTH_M * CUTOFF_SIGMA
INTENSITY_THRESHOLD = 0.02

hex_path = Path("./public/heatmaps/hex_centers_3000m.json")
with hex_path.open("r", encoding="utf-8") as f:
    hex_data = json.load(f)

hex_centers_pdf = pd.DataFrame(hex_data["cells"])
hex_centers_sdf = spark.createDataFrame(hex_centers_pdf)

sheet_names = list(all_sheets.keys())
accident_source = spark.createDataFrame(all_sheets[sheet_names[2]])

severity_weight_expr = (
    F.when(F.col("severity") == 1, F.lit(5.0))
     .when(F.col("severity") == 2, F.lit(3.0))
     .when(F.col("severity") == 3, F.lit(1.0))
     .otherwise(F.lit(1.0))
)

accident_2018_m = (
    accident_source
    .select("XGCSWGS84", "YGCSWGS84", "severity")
    .withColumn("longitude", F.col("XGCSWGS84").cast("double"))
    .withColumn("latitude", F.col("YGCSWGS84").cast("double"))
    .filter(F.col("longitude").isNotNull() & F.col("latitude").isNotNull())
    .withColumn("x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
    .withColumn("severity_weight", severity_weight_expr)
    .filter(F.col("severity_weight").isNotNull())
)

hex_centers_m = (
    hex_centers_sdf
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("center_x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("center_y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
)

hex_centers_broadcast = F.broadcast(hex_centers_m.select("hex_row", "hex_col", "longitude", "latitude", "center_x_m", "center_y_m"))

kde_contrib = (
    accident_2018_m.alias("a")
    .crossJoin(hex_centers_broadcast.alias("h"))
    .withColumn(
        "distance_m",
        F.sqrt(
            F.pow(F.col("a.x_m") - F.col("h.center_x_m"), 2) +
            F.pow(F.col("a.y_m") - F.col("h.center_y_m"), 2)
        )
    )
    .filter(F.col("distance_m") <= F.lit(CUTOFF_M))
    .withColumn(
        "distance_weight",
        F.exp(-(F.col("distance_m") * F.col("distance_m")) / F.lit(2.0 * BANDWIDTH_M * BANDWIDTH_M))
    )
    .withColumn("contribution", F.col("distance_weight") * F.col("a.severity_weight"))
)

raw_intensity = (
    kde_contrib
    .groupBy("h.hex_row", "h.hex_col", "h.longitude", "h.latitude", "h.center_x_m", "h.center_y_m")
    .agg(
        F.sum("contribution").alias("raw_intensity"),
        F.count(F.lit(1)).alias("support_points")
    )
)

max_raw = raw_intensity.agg(F.max("raw_intensity").alias("max_raw")).collect()[0]["max_raw"]
if max_raw is None or max_raw == 0:
    max_raw = 1.0

heatmap_2018 = (
    raw_intensity
    .withColumn("heatmap_intensity", F.col("raw_intensity") / F.lit(float(max_raw)))
    .orderBy(F.desc("raw_intensity"))
)

heatmap_2018_sparse = heatmap_2018.filter(F.col("heatmap_intensity") >= F.lit(INTENSITY_THRESHOLD))

rows = heatmap_2018_sparse.select("longitude", "latitude", "raw_intensity", "heatmap_intensity", "support_points").toLocalIterator()
records = [r.asDict(recursive=True) for r in rows]
payload = {"year": 2018, "bandwidth_m": BANDWIDTH_M, "intensity_threshold": INTENSITY_THRESHOLD, "cells": records}

out_path = "./public/heatmaps/heatmap_2018.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=True)

print(f"Wrote {out_path} with {len(records)} cells")

## Step 3.5.: Generate 2019 KDE Heatmap

Compute kernel density estimation for 2019 accidents on the 3000m hex grid using identical parameters for consistency across all years.

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

EARTH_RADIUS_M = 6378137.0
BANDWIDTH_M = 3000.0
CUTOFF_SIGMA = 3.0
CUTOFF_M = BANDWIDTH_M * CUTOFF_SIGMA
INTENSITY_THRESHOLD = 0.02

hex_path = Path("./public/heatmaps/hex_centers_3000m.json")
with hex_path.open("r", encoding="utf-8") as f:
    hex_data = json.load(f)

hex_centers_pdf = pd.DataFrame(hex_data["cells"])
hex_centers_sdf = spark.createDataFrame(hex_centers_pdf)

sheet_names = list(all_sheets.keys())
accident_source = spark.createDataFrame(all_sheets[sheet_names[3]])

severity_weight_expr = (
    F.when(F.col("severity") == 1, F.lit(5.0))
     .when(F.col("severity") == 2, F.lit(3.0))
     .when(F.col("severity") == 3, F.lit(1.0))
     .otherwise(F.lit(1.0))
)

accident_2019_m = (
    accident_source
    .select("XGCSWGS84", "YGCSWGS84", "severity")
    .withColumn("longitude", F.col("XGCSWGS84").cast("double"))
    .withColumn("latitude", F.col("YGCSWGS84").cast("double"))
    .filter(F.col("longitude").isNotNull() & F.col("latitude").isNotNull())
    .withColumn("x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
    .withColumn("severity_weight", severity_weight_expr)
    .filter(F.col("severity_weight").isNotNull())
)

hex_centers_m = (
    hex_centers_sdf
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("center_x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("center_y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
)

hex_centers_broadcast = F.broadcast(hex_centers_m.select("hex_row", "hex_col", "longitude", "latitude", "center_x_m", "center_y_m"))

kde_contrib = (
    accident_2019_m.alias("a")
    .crossJoin(hex_centers_broadcast.alias("h"))
    .withColumn(
        "distance_m",
        F.sqrt(
            F.pow(F.col("a.x_m") - F.col("h.center_x_m"), 2) +
            F.pow(F.col("a.y_m") - F.col("h.center_y_m"), 2)
        )
    )
    .filter(F.col("distance_m") <= F.lit(CUTOFF_M))
    .withColumn(
        "distance_weight",
        F.exp(-(F.col("distance_m") * F.col("distance_m")) / F.lit(2.0 * BANDWIDTH_M * BANDWIDTH_M))
    )
    .withColumn("contribution", F.col("distance_weight") * F.col("a.severity_weight"))
)

raw_intensity = (
    kde_contrib
    .groupBy("h.hex_row", "h.hex_col", "h.longitude", "h.latitude", "h.center_x_m", "h.center_y_m")
    .agg(
        F.sum("contribution").alias("raw_intensity"),
        F.count(F.lit(1)).alias("support_points")
    )
)

max_raw = raw_intensity.agg(F.max("raw_intensity").alias("max_raw")).collect()[0]["max_raw"]
if max_raw is None or max_raw == 0:
    max_raw = 1.0

heatmap_2019 = (
    raw_intensity
    .withColumn("heatmap_intensity", F.col("raw_intensity") / F.lit(float(max_raw)))
    .orderBy(F.desc("raw_intensity"))
)

heatmap_2019_sparse = heatmap_2019.filter(F.col("heatmap_intensity") >= F.lit(INTENSITY_THRESHOLD))

rows = heatmap_2019_sparse.select("longitude", "latitude", "raw_intensity", "heatmap_intensity", "support_points").toLocalIterator()
records = [r.asDict(recursive=True) for r in rows]
payload = {"year": 2019, "bandwidth_m": BANDWIDTH_M, "intensity_threshold": INTENSITY_THRESHOLD, "cells": records}

out_path = "./public/heatmaps/heatmap_2019.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=True)

print(f"Wrote {out_path} with {len(records)} cells")

## Step 3.6.: Generate 2020 KDE Heatmap

Compute kernel density estimation for 2020 accidents on the 3000m hex grid using identical parameters for consistency across all years.

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

EARTH_RADIUS_M = 6378137.0
BANDWIDTH_M = 3000.0
CUTOFF_SIGMA = 3.0
CUTOFF_M = BANDWIDTH_M * CUTOFF_SIGMA
INTENSITY_THRESHOLD = 0.02

hex_path = Path("./public/heatmaps/hex_centers_3000m.json")
with hex_path.open("r", encoding="utf-8") as f:
    hex_data = json.load(f)

hex_centers_pdf = pd.DataFrame(hex_data["cells"])
hex_centers_sdf = spark.createDataFrame(hex_centers_pdf)

sheet_names = list(all_sheets.keys())
accident_source = spark.createDataFrame(all_sheets[sheet_names[4]])

severity_weight_expr = (
    F.when(F.col("severity") == 1, F.lit(5.0))
     .when(F.col("severity") == 2, F.lit(3.0))
     .when(F.col("severity") == 3, F.lit(1.0))
     .otherwise(F.lit(1.0))
)

accident_2020_m = (
    accident_source
    .select("XGCSWGS84", "YGCSWGS84", "severity")
    .withColumn("longitude", F.col("XGCSWGS84").cast("double"))
    .withColumn("latitude", F.col("YGCSWGS84").cast("double"))
    .filter(F.col("longitude").isNotNull() & F.col("latitude").isNotNull())
    .withColumn("x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
    .withColumn("severity_weight", severity_weight_expr)
    .filter(F.col("severity_weight").isNotNull())
)

hex_centers_m = (
    hex_centers_sdf
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("center_x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("center_y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
)

hex_centers_broadcast = F.broadcast(hex_centers_m.select("hex_row", "hex_col", "longitude", "latitude", "center_x_m", "center_y_m"))

kde_contrib = (
    accident_2020_m.alias("a")
    .crossJoin(hex_centers_broadcast.alias("h"))
    .withColumn(
        "distance_m",
        F.sqrt(
            F.pow(F.col("a.x_m") - F.col("h.center_x_m"), 2) +
            F.pow(F.col("a.y_m") - F.col("h.center_y_m"), 2)
        )
    )
    .filter(F.col("distance_m") <= F.lit(CUTOFF_M))
    .withColumn(
        "distance_weight",
        F.exp(-(F.col("distance_m") * F.col("distance_m")) / F.lit(2.0 * BANDWIDTH_M * BANDWIDTH_M))
    )
    .withColumn("contribution", F.col("distance_weight") * F.col("a.severity_weight"))
)

raw_intensity = (
    kde_contrib
    .groupBy("h.hex_row", "h.hex_col", "h.longitude", "h.latitude", "h.center_x_m", "h.center_y_m")
    .agg(
        F.sum("contribution").alias("raw_intensity"),
        F.count(F.lit(1)).alias("support_points")
    )
)

max_raw = raw_intensity.agg(F.max("raw_intensity").alias("max_raw")).collect()[0]["max_raw"]
if max_raw is None or max_raw == 0:
    max_raw = 1.0

heatmap_2020 = (
    raw_intensity
    .withColumn("heatmap_intensity", F.col("raw_intensity") / F.lit(float(max_raw)))
    .orderBy(F.desc("raw_intensity"))
)

heatmap_2020_sparse = heatmap_2020.filter(F.col("heatmap_intensity") >= F.lit(INTENSITY_THRESHOLD))

rows = heatmap_2020_sparse.select("longitude", "latitude", "raw_intensity", "heatmap_intensity", "support_points").toLocalIterator()
records = [r.asDict(recursive=True) for r in rows]
payload = {"year": 2020, "bandwidth_m": BANDWIDTH_M, "intensity_threshold": INTENSITY_THRESHOLD, "cells": records}

out_path = "./public/heatmaps/heatmap_2020.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=True)

print(f"Wrote {out_path} with {len(records)} cells")

## Step 3.7.: Generate 2021 KDE Heatmap

Compute kernel density estimation for 2021 accidents on the 3000m hex grid using identical parameters for consistency across all years.

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

EARTH_RADIUS_M = 6378137.0
BANDWIDTH_M = 3000.0
CUTOFF_SIGMA = 3.0
CUTOFF_M = BANDWIDTH_M * CUTOFF_SIGMA
INTENSITY_THRESHOLD = 0.02

hex_path = Path("./public/heatmaps/hex_centers_3000m.json")
with hex_path.open("r", encoding="utf-8") as f:
    hex_data = json.load(f)

hex_centers_pdf = pd.DataFrame(hex_data["cells"])
hex_centers_sdf = spark.createDataFrame(hex_centers_pdf)

sheet_names = list(all_sheets.keys())
accident_source = spark.createDataFrame(all_sheets[sheet_names[5]])

severity_weight_expr = (
    F.when(F.col("severity") == 1, F.lit(5.0))
     .when(F.col("severity") == 2, F.lit(3.0))
     .when(F.col("severity") == 3, F.lit(1.0))
     .otherwise(F.lit(1.0))
)

accident_2021_m = (
    accident_source
    .select("XGCSWGS84", "YGCSWGS84", "severity")
    .withColumn("longitude", F.col("XGCSWGS84").cast("double"))
    .withColumn("latitude", F.col("YGCSWGS84").cast("double"))
    .filter(F.col("longitude").isNotNull() & F.col("latitude").isNotNull())
    .withColumn("x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
    .withColumn("severity_weight", severity_weight_expr)
    .filter(F.col("severity_weight").isNotNull())
)

hex_centers_m = (
    hex_centers_sdf
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("center_x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("center_y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
)

hex_centers_broadcast = F.broadcast(hex_centers_m.select("hex_row", "hex_col", "longitude", "latitude", "center_x_m", "center_y_m"))

kde_contrib = (
    accident_2021_m.alias("a")
    .crossJoin(hex_centers_broadcast.alias("h"))
    .withColumn(
        "distance_m",
        F.sqrt(
            F.pow(F.col("a.x_m") - F.col("h.center_x_m"), 2) +
            F.pow(F.col("a.y_m") - F.col("h.center_y_m"), 2)
        )
    )
    .filter(F.col("distance_m") <= F.lit(CUTOFF_M))
    .withColumn(
        "distance_weight",
        F.exp(-(F.col("distance_m") * F.col("distance_m")) / F.lit(2.0 * BANDWIDTH_M * BANDWIDTH_M))
    )
    .withColumn("contribution", F.col("distance_weight") * F.col("a.severity_weight"))
)

raw_intensity = (
    kde_contrib
    .groupBy("h.hex_row", "h.hex_col", "h.longitude", "h.latitude", "h.center_x_m", "h.center_y_m")
    .agg(
        F.sum("contribution").alias("raw_intensity"),
        F.count(F.lit(1)).alias("support_points")
    )
)

max_raw = raw_intensity.agg(F.max("raw_intensity").alias("max_raw")).collect()[0]["max_raw"]
if max_raw is None or max_raw == 0:
    max_raw = 1.0

heatmap_2021 = (
    raw_intensity
    .withColumn("heatmap_intensity", F.col("raw_intensity") / F.lit(float(max_raw)))
    .orderBy(F.desc("raw_intensity"))
)

heatmap_2021_sparse = heatmap_2021.filter(F.col("heatmap_intensity") >= F.lit(INTENSITY_THRESHOLD))

rows = heatmap_2021_sparse.select("longitude", "latitude", "raw_intensity", "heatmap_intensity", "support_points").toLocalIterator()
records = [r.asDict(recursive=True) for r in rows]
payload = {"year": 2021, "bandwidth_m": BANDWIDTH_M, "intensity_threshold": INTENSITY_THRESHOLD, "cells": records}

out_path = "./public/heatmaps/heatmap_2021.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=True)

print(f"Wrote {out_path} with {len(records)} cells")

## Step 3.8.: Generate 2022 KDE Heatmap

Compute kernel density estimation for 2022 accidents on the 3000m hex grid using identical parameters for consistency across all years.

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

EARTH_RADIUS_M = 6378137.0
BANDWIDTH_M = 3000.0
CUTOFF_SIGMA = 3.0
CUTOFF_M = BANDWIDTH_M * CUTOFF_SIGMA
INTENSITY_THRESHOLD = 0.02

hex_path = Path("./public/heatmaps/hex_centers_3000m.json")
with hex_path.open("r", encoding="utf-8") as f:
    hex_data = json.load(f)

hex_centers_pdf = pd.DataFrame(hex_data["cells"])
hex_centers_sdf = spark.createDataFrame(hex_centers_pdf)

sheet_names = list(all_sheets.keys())
accident_source = spark.createDataFrame(all_sheets[sheet_names[6]])

severity_weight_expr = (
    F.when(F.col("severity") == 1, F.lit(5.0))
     .when(F.col("severity") == 2, F.lit(3.0))
     .when(F.col("severity") == 3, F.lit(1.0))
     .otherwise(F.lit(1.0))
)

accident_2022_m = (
    accident_source
    .select("XGCSWGS84", "YGCSWGS84", "severity")
    .withColumn("longitude", F.col("XGCSWGS84").cast("double"))
    .withColumn("latitude", F.col("YGCSWGS84").cast("double"))
    .filter(F.col("longitude").isNotNull() & F.col("latitude").isNotNull())
    .withColumn("x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
    .withColumn("severity_weight", severity_weight_expr)
    .filter(F.col("severity_weight").isNotNull())
)

hex_centers_m = (
    hex_centers_sdf
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("center_x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("center_y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
)

hex_centers_broadcast = F.broadcast(hex_centers_m.select("hex_row", "hex_col", "longitude", "latitude", "center_x_m", "center_y_m"))

kde_contrib = (
    accident_2022_m.alias("a")
    .crossJoin(hex_centers_broadcast.alias("h"))
    .withColumn(
        "distance_m",
        F.sqrt(
            F.pow(F.col("a.x_m") - F.col("h.center_x_m"), 2) +
            F.pow(F.col("a.y_m") - F.col("h.center_y_m"), 2)
        )
    )
    .filter(F.col("distance_m") <= F.lit(CUTOFF_M))
    .withColumn(
        "distance_weight",
        F.exp(-(F.col("distance_m") * F.col("distance_m")) / F.lit(2.0 * BANDWIDTH_M * BANDWIDTH_M))
    )
    .withColumn("contribution", F.col("distance_weight") * F.col("a.severity_weight"))
)

raw_intensity = (
    kde_contrib
    .groupBy("h.hex_row", "h.hex_col", "h.longitude", "h.latitude", "h.center_x_m", "h.center_y_m")
    .agg(
        F.sum("contribution").alias("raw_intensity"),
        F.count(F.lit(1)).alias("support_points")
    )
)

max_raw = raw_intensity.agg(F.max("raw_intensity").alias("max_raw")).collect()[0]["max_raw"]
if max_raw is None or max_raw == 0:
    max_raw = 1.0

heatmap_2022 = (
    raw_intensity
    .withColumn("heatmap_intensity", F.col("raw_intensity") / F.lit(float(max_raw)))
    .orderBy(F.desc("raw_intensity"))
)

heatmap_2022_sparse = heatmap_2022.filter(F.col("heatmap_intensity") >= F.lit(INTENSITY_THRESHOLD))

rows = heatmap_2022_sparse.select("longitude", "latitude", "raw_intensity", "heatmap_intensity", "support_points").toLocalIterator()
records = [r.asDict(recursive=True) for r in rows]
payload = {"year": 2022, "bandwidth_m": BANDWIDTH_M, "intensity_threshold": INTENSITY_THRESHOLD, "cells": records}

out_path = "./public/heatmaps/heatmap_2022.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=True)

print(f"Wrote {out_path} with {len(records)} cells")

## Step 3.9.: Generate 2023 KDE Heatmap

Compute kernel density estimation for 2023 accidents on the 3000m hex grid using identical parameters for consistency across all years.

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

EARTH_RADIUS_M = 6378137.0
BANDWIDTH_M = 3000.0
CUTOFF_SIGMA = 3.0
CUTOFF_M = BANDWIDTH_M * CUTOFF_SIGMA
INTENSITY_THRESHOLD = 0.02

hex_path = Path("./public/heatmaps/hex_centers_3000m.json")
with hex_path.open("r", encoding="utf-8") as f:
    hex_data = json.load(f)

hex_centers_pdf = pd.DataFrame(hex_data["cells"])
hex_centers_sdf = spark.createDataFrame(hex_centers_pdf)

sheet_names = list(all_sheets.keys())
accident_source = spark.createDataFrame(all_sheets[sheet_names[7]])

severity_weight_expr = (
    F.when(F.col("severity") == 1, F.lit(5.0))
     .when(F.col("severity") == 2, F.lit(3.0))
     .when(F.col("severity") == 3, F.lit(1.0))
     .otherwise(F.lit(1.0))
)

accident_2023_m = (
    accident_source
    .select("XGCSWGS84", "YGCSWGS84", "severity")
    .withColumn("longitude", F.col("XGCSWGS84").cast("double"))
    .withColumn("latitude", F.col("YGCSWGS84").cast("double"))
    .filter(F.col("longitude").isNotNull() & F.col("latitude").isNotNull())
    .withColumn("x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
    .withColumn("severity_weight", severity_weight_expr)
    .filter(F.col("severity_weight").isNotNull())
)

hex_centers_m = (
    hex_centers_sdf
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("center_x_m", F.col("longitude") * F.lit(EARTH_RADIUS_M * math.pi / 180.0))
    .withColumn("center_y_m", F.log(F.tan(F.lit(math.pi / 4.0) + F.radians(F.col("latitude")) / 2.0)) * F.lit(EARTH_RADIUS_M))
)

hex_centers_broadcast = F.broadcast(hex_centers_m.select("hex_row", "hex_col", "longitude", "latitude", "center_x_m", "center_y_m"))

kde_contrib = (
    accident_2023_m.alias("a")
    .crossJoin(hex_centers_broadcast.alias("h"))
    .withColumn(
        "distance_m",
        F.sqrt(
            F.pow(F.col("a.x_m") - F.col("h.center_x_m"), 2) +
            F.pow(F.col("a.y_m") - F.col("h.center_y_m"), 2)
        )
    )
    .filter(F.col("distance_m") <= F.lit(CUTOFF_M))
    .withColumn(
        "distance_weight",
        F.exp(-(F.col("distance_m") * F.col("distance_m")) / F.lit(2.0 * BANDWIDTH_M * BANDWIDTH_M))
    )
    .withColumn("contribution", F.col("distance_weight") * F.col("a.severity_weight"))
)

raw_intensity = (
    kde_contrib
    .groupBy("h.hex_row", "h.hex_col", "h.longitude", "h.latitude", "h.center_x_m", "h.center_y_m")
    .agg(
        F.sum("contribution").alias("raw_intensity"),
        F.count(F.lit(1)).alias("support_points")
    )
)

max_raw = raw_intensity.agg(F.max("raw_intensity").alias("max_raw")).collect()[0]["max_raw"]
if max_raw is None or max_raw == 0:
    max_raw = 1.0

heatmap_2023 = (
    raw_intensity
    .withColumn("heatmap_intensity", F.col("raw_intensity") / F.lit(float(max_raw)))
    .orderBy(F.desc("raw_intensity"))
)

heatmap_2023_sparse = heatmap_2023.filter(F.col("heatmap_intensity") >= F.lit(INTENSITY_THRESHOLD))

rows = heatmap_2023_sparse.select("longitude", "latitude", "raw_intensity", "heatmap_intensity", "support_points").toLocalIterator()
records = [r.asDict(recursive=True) for r in rows]
payload = {"year": 2023, "bandwidth_m": BANDWIDTH_M, "intensity_threshold": INTENSITY_THRESHOLD, "cells": records}

out_path = "./public/heatmaps/heatmap_2023.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=True)

print(f"Wrote {out_path} with {len(records)} cells")

## Step 4: Build Severity-by-Road-Surface Yearly Counts

For every severity and within each severity, for each road surface type, we find the yearly count of accidents. But since we have some missing data, we normalize the count, i.e. convert to percentages.The result is stored in nested JSON structure with counts per: `severity_death`,`roadSurfaceCondition`, `year`

File is saved in 
`public/heatmaps/accident_counts_by_year_roadSurface_severity.json`
`public/heatmaps/accident_counts_by_year_roadSurface_severity_percent.json`

### What it does

For each yearly Spark DataFrame, we groups accidents by `roadSurfaceCondition` and `severity` and counts accidents in each group

It also fills in missing year values with `0` so every road surface has the same year keys.

### Shape:
```json
[
  {
    "severity_death": 1,
    "accident_detail": [
      {
        "roadSurfaceCondition": 0,
        "year_count": {
          "2016": 71.7,
          "2017": 71.8,
        }
      },
      {
        "roadSurfaceCondition": 1,
        "year_count": {
          "2016": 26.2,
          "2017": 26.1,
        }
      }, ...
    ]
  }
]

In [ ]:
import json
from pathlib import Path
from pyspark.sql import functions as F

# Produce new structure: per severity, accident counts per roadSurface per year (raw counts)
year_names = list(all_sheets.keys())

# initialize map: severity -> road -> {year: count}
severity_map = {1: {0: {}, 1: {}, 2: {}}, 2: {0: {}, 1: {}, 2: {}}, 3: {0: {}, 1: {}, 2: {}}}

for year_index, sdf in enumerate(spark_dataframes):
    # derive numeric year similar to earlier logic
    year_value = int(year_names[year_index]) if year_index < len(year_names) and str(year_names[year_index]).isdigit() else 2016 + year_index

    grouped = (
        sdf
        .select(
            F.col("roadSurfaceCondition").cast("int").alias("roadSurfaceCondition"),
            F.col("severity").cast("int").alias("severity")
        )
        .groupBy("roadSurfaceCondition", "severity")
        .agg(F.count(F.lit(1)).alias("accident_count"))
        .orderBy("roadSurfaceCondition", "severity")
    )

    rows = grouped.collect()

    # populate severity_map with counts for this year
    for row in rows:
        road_surface = int(row["roadSurfaceCondition"]) if row["roadSurfaceCondition"] is not None else None
        severity = int(row["severity"]) if row["severity"] is not None else None
        count = int(row["accident_count"]) if row["accident_count"] is not None else 0
        if severity in severity_map and road_surface in severity_map[severity]:
            severity_map[severity][road_surface][str(year_value)] = count

# ensure zeros for missing year entries
# build numeric years_list from sheet names or index
years_list = [int(name) if str(name).isdigit() else 2016 + i for i, name in enumerate(year_names)]
for sev in [1,2,3]:
    for road in [0,1,2]:
        for y in years_list:
            severity_map[sev][road].setdefault(str(y), 0)

# build the final payload (raw counts)
output = []
for sev in [1,2,3]:
    accident_detail = []
    for road in sorted(severity_map[sev].keys()):
        # sort year_count keys for consistent ordering
        year_count = {yr: severity_map[sev][road][yr] for yr in sorted(severity_map[sev][road].keys())}
        accident_detail.append({
            "roadSurfaceCondition": road,
            "year_count": year_count
        })
    output.append({
        "severity_death": sev,
        "accident_detail": accident_detail
    })

output_path = Path("./public/heatmaps/accident_counts_by_year_roadSurface_severity.json")
with output_path.open("w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=True, indent=2)

print(f"Wrote {output_path} with {len(output)} severity entries")
print(json.dumps(output[:1], indent=2)[:1000])

# --- New: compute per-year percentages across roadSurface conditions and write new file ---
percent_output = []
for sev_entry in output:
    sev = sev_entry["severity_death"]
    # collect years (assume same years for all roadSurface entries)
    years = []
    if sev_entry["accident_detail"]:
        years = sorted(list(sev_entry["accident_detail"][0]["year_count"].keys()))

    # compute totals per year across road surfaces
    totals_by_year = {y: 0 for y in years}
    for detail in sev_entry["accident_detail"]:
        for y, c in detail["year_count"].items():
            totals_by_year[y] = totals_by_year.get(y, 0) + int(c)

    # build percent details
    percent_details = []
    for detail in sev_entry["accident_detail"]:
        road = detail["roadSurfaceCondition"]
        year_count_pct = {}
        for y in years:
            total = totals_by_year.get(y, 0)
            raw = int(detail["year_count"].get(y, 0))
            pct = round((raw / total) * 100, 1) if total > 0 else 0.0
            year_count_pct[y] = pct
        percent_details.append({
            "roadSurfaceCondition": road,
            "year_count": year_count_pct
        })

    percent_output.append({
        "severity_death": sev,
        "accident_detail": percent_details
    })

out_percent_path = Path("./public/heatmaps/accident_counts_by_year_roadSurface_severity_percent.json")
with out_percent_path.open("w", encoding="utf-8") as f:
    json.dump(percent_output, f, ensure_ascii=True, indent=2)

print(f"Wrote percent file: {out_percent_path} with {len(percent_output)} severity entries")

## Step 5.1.: Road Surface 2 Severity-by-Year Aggregation (Raw Counts)

For only **road surface condition 2 (slippery/snowy roads)**, we shall count how many accidents occured for each of the severity level. It produces a raw accident counts (not percentages) and writes the result to a JSON file.

## Output File
`roadSurface2_accident_counts_by_severity_year.json`

In [ ]:
import json
from pathlib import Path
from pyspark.sql import functions as F

# Group by roadSurfaceCondition == 2, counts per severity per year
year_names_local = globals().get('year_names', list(all_sheets.keys()))
years_list = [int(name) if str(name).isdigit() else 2016 + i for i, name in enumerate(year_names_local)]

# initialize map severity -> {year: count}
result_map = {1: {}, 2: {}, 3: {}}

for idx, sdf in enumerate(spark_dataframes):
    year = years_list[idx] if idx < len(years_list) else 2016 + idx
    grouped = (
        sdf
        .select(
            F.col("roadSurfaceCondition").cast("int").alias("roadSurfaceCondition"),
            F.col("severity").cast("int").alias("severity")
        )
        .filter(F.col("roadSurfaceCondition") == F.lit(2))
        .groupBy("severity")
        .agg(F.count(F.lit(1)).alias("accident_count"))
    )

    rows = grouped.collect()
    for row in rows:
        sev = int(row["severity"]) if row["severity"] is not None else None
        cnt = int(row["accident_count"]) if row["accident_count"] is not None else 0
        if sev in result_map:
            result_map[sev][str(year)] = cnt

# ensure zeros for missing year entries
for sev in [1,2,3]:
    for y in years_list:
        result_map[sev].setdefault(str(y), 0)

# Build payload
payload = {
    "roadSurfaceCondition": 2,
    "accident_detail": []
}
for sev in [1,2,3]:
    year_count = {yr: result_map[sev][yr] for yr in sorted(result_map[sev].keys())}
    payload["accident_detail"].append({
        "severity": sev,
        "year_count": year_count
    })

out_path = Path("./public/heatmaps/roadSurface2_accident_counts_by_severity_year.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with out_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=True, indent=2)

print(f"Wrote {out_path} with {len(payload['accident_detail'])} severities")
print(json.dumps(payload, indent=2)[:1000])

## Step 5.2.: Road Surface 2 Severity-by-Year Normalization (Percentage Conversion)

Based on the `roadSurface2_accident_counts_by_severity_year.json` , we shall now normalize, or convert the count into percentages.

## Output File
`roadSurface2_accident_counts_by_severity_year_percent.json`

In [ ]:
import json
from pathlib import Path

# Normalize roadSurface2 JSON - convert counts to percentages and add total
input_path = Path("./public/heatmaps/roadSurface2_accident_counts_by_severity_year.json")
with input_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

# Normalize each severity
normalized_detail = []
for detail in data["accident_detail"]:
    severity = detail["severity"]
    year_count = detail["year_count"]
    
    # Calculate total across all years
    total = sum(year_count.values())
    
    # Convert to percentages
    year_count_pct = {year: (count / total * 100) if total > 0 else 0 for year, count in year_count.items()}
    
    normalized_detail.append({
        "severity": severity,
        "total": total,
        "year_count": year_count_pct
    })

# Build normalized payload
normalized_payload = {
    "roadSurfaceCondition": 2,
    "accident_detail": normalized_detail
}

# Save normalized JSON
output_path = Path("./public/heatmaps/roadSurface2_accident_counts_by_severity_year_percent.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    json.dump(normalized_payload, f, ensure_ascii=True, indent=2)

print(f"Wrote {output_path}")
print(json.dumps(normalized_payload, indent=2)[:1000])

# Step 6: State-Level Accident Aggregation (Yearly Counts)

## Purpose
This cell aggregates accident data at the **state level** to count how many total accidents occurred in each of Germany's 16 states per year. It creates a year-major JSON structure that groups accident counts by state within each year.

## Output File
`accidents_by_year_state.json`

### **Build Output Structure**
   Creates a year-major array with this shape:
   ```json
   [
     {
       "year": 2016,
       "accident_data": {
         "Schleswig-Holstein": 1234,
         "Hamburg": 567,
         "Lower Saxony": 3456,
         "Bremen": 123,
         ...
         "Thuringia": 890
       }
     },
     {
       "year": 2017,
       "accident_data": { ... }
     },
     ...
   ]

In [ ]:
import json
from pathlib import Path
from pyspark.sql import functions as F

# Count accidents per year per state and export as JSON
# Output shape:
# [
#   {
#     "year": 2016,
#     "accident_data": {
#       "Schleswig-Holstein": 123,
#       "Hamburg": 456,
#       ...
#     }
#   },
#   ...
# ]

STATE_CODE_TO_NAME = {
    "01": "Schleswig-Holstein",
    "02": "Hamburg",
    "03": "Lower Saxony",
    "04": "Bremen",
    "05": "North Rhine-Westphalia",
    "06": "Hesse",
    "07": "Rhineland-Palatinate",
    "08": "Baden-Wuerttemberg",
    "09": "Bavaria",
    "10": "Saarland",
    "11": "Berlin",
    "12": "Brandenburg",
    "13": "Mecklenburg-Western Pomerania",
    "14": "Saxony",
    "15": "Saxony-Anhalt",
    "16": "Thuringia",
}

year_names_local = globals().get("year_names", list(all_sheets.keys()))
years_list = [int(name) if str(name).isdigit() else 2016 + i for i, name in enumerate(year_names_local)]
state_names = list(STATE_CODE_TO_NAME.values())

result = []

for idx, sdf in enumerate(spark_dataframes):
    year = years_list[idx] if idx < len(years_list) else 2016 + idx

    grouped = (
        sdf.select(F.col("stateCode").cast("string").alias("stateCode"))
        .groupBy("stateCode")
        .agg(F.count(F.lit(1)).alias("accident_count"))
    )

    accident_data = {state_name: 0 for state_name in state_names}

    for row in grouped.collect():
        state_code = row["stateCode"]
        accident_count = int(row["accident_count"] or 0)

        if state_code is not None:
            state_code = str(state_code).zfill(2)
            state_name = STATE_CODE_TO_NAME.get(state_code)
            if state_name is not None:
                accident_data[state_name] = accident_count

    result.append({
        "year": year,
        "accident_data": accident_data,
    })

output_path = Path("./public/heatmaps/accidents_by_year_state.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=True, indent=2)

print(f"Wrote {output_path} with {len(result)} years")
print(json.dumps(result[:1], indent=2)[:1200])